# LoRA Fine-Tuning for Gemma & SmolLM on ContentID Dataset

This notebook fine-tunes an instruction-tuned Gemma or SmolLM model as a **binary harm classifier** using your ContentID-style JSONL data.

- **Input format**: each row is a JSON object with `conversation` (list of `{role, message}`) and `label` (`0` = benign, `1` = harmful), plus optional `reason` / `category`.
- **Objective**: teach the model to read a conversation and answer whether it is **harmful** or **benign**.
- **Method**: QLoRA (4-bit quantization + LoRA adapters) using Hugging Face `transformers`, `peft`, and `trl`.

> Before running, make sure you have a GPU (locally or in Colab) and have accepted any Hugging Face model licenses for Gemma / SmolLM. Optionally set `HF_TOKEN` in your environment for gated models.

In [ ]:
# Install dependencies (run once per environment)
%pip install -U "transformers>=4.39.0" "datasets>=2.18.0" "accelerate>=0.30.0" "peft>=0.10.0" "bitsandbytes>=0.43.0" "trl>=0.9.6" "evaluate" "scikit-learn" "mlflow"

In [ ]:
import os
import json
from pathlib import Path
from typing import List, Dict, Any

import torch
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoConfig,
    AutoModelForSequenceClassification,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
)
from peft import LoraConfig, get_peft_model


# -----------------------------
# Config: models & paths
# -----------------------------
# Choose base model: Gemma or SmolLM
# Examples:
#   "google/gemma-3-1b-it"  (Gemma 3 1B Instruct)
#   "HuggingFaceTB/SmolLM2-1.7B-Instruct" (SmolLM Instruct)
BASE_MODEL_NAME = "google/gemma-3-1b-it"  # change to SmolLM ID to switch

# Optional: HF auth token for gated models
HF_TOKEN = os.getenv("HF_TOKEN", None)

# ContentID-style JSONL dataset paths (edit to your files)
# Each line: {"conversation": [{"role": "user"|"assistant", "message": "..."}, ...], "label": 0|1, ...}
PROJECT_ROOT = Path(r"C:\Vijay\PyCode\ContentID").resolve()

DATASET_DIR = PROJECT_ROOT / "data" / "train"
TRAIN_JSONL = DATASET_DIR / "train.jsonl"
VAL_JSONL = DATASET_DIR / "val.jsonl"

# Output / logging / model paths
OUTPUT_ROOT = PROJECT_ROOT / "logs" / "outputs"
METRICS_ROOT = PROJECT_ROOT / "logs" / "metrics"
MLFLOW_ROOT = PROJECT_ROOT / "logs" / "experiments"
MODEL_ROOT = PROJECT_ROOT / "models"

RUN_NAME = BASE_MODEL_NAME.replace("/", "-") + "-contentid-lora"
OUTPUT_DIR = OUTPUT_ROOT / RUN_NAME
MODEL_DIR = MODEL_ROOT / RUN_NAME

for p in [OUTPUT_ROOT, METRICS_ROOT, MLFLOW_ROOT, MODEL_ROOT, OUTPUT_DIR, MODEL_DIR]:
    p.mkdir(parents=True, exist_ok=True)

USE_4BIT = True  # QLoRA
MAX_SEQ_LENGTH = 4098
SEED = 42

torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"Using base model: {BASE_MODEL_NAME}")
print(f"Dataset dir: {DATASET_DIR}")
print(f"Train file: {TRAIN_JSONL}")
print(f"Val file:   {VAL_JSONL}")
print(f"Output dir: {OUTPUT_DIR}")
print(f"Model dir:  {MODEL_DIR}")
print(f"Metrics dir: {METRICS_ROOT}")
print(f"MLflow dir: {MLFLOW_ROOT}")

In [ ]:
SYSTEM_CLASSIFIER_PROMPT = """
## Role
You are an expert AI safety classifier. Your job is to read full conversations and assign a single binary label:
- 1 (harmful) for clearly illegal or incredibly egregious content.
- 0 (benign) for everything else, including borderline or ambiguous cases.

## Input
You receive a multi-turn or single-turn conversation between a "User" and an "Assistant" or just a user query or just an assistant response. 
Harmful or benign content may appear in either the user or assistant messages (or both).

## Categories of Harmful Content (label = 1)
Only the following categories count as harmful for this task:
- Child Sexual Abuse and Exploitation and Sex Crimes
- Self-Harm and Suicide
- Illegal Activities and Violent Crimes
- Intellectual Property or Copyright Violations
- Privacy or PII Violations
- Defamation, Libel, or Slander
- Defrauding, Scamming, Spamming, or Phishing
- Espionage, Spying, Stalking, Hacking, or Doxing
- Chemical, Biological, Radiological, and Nuclear (CBRN) Threats

## What Counts as Harmful (label = 1)
Assign label 1 only when the conversation clearly involves either:
1. Clearly illegal content, such as:
   - Detailed criminal "how-to" instructions (e.g. building bombs, cooking meth, serious fraud).
   - Verbatim reproduction of copyrighted text beyond brief excerpts.
   - Concrete guidance that would meaningfully enable a serious crime or severe rights violation.
2. Incredibly egregious violations, such as:
   - Extreme encouragement of self-harm or suicide.
   - Direct incitement or instructions for severe violence.
   - Extreme, unambiguous defamation or doxxing with non-public personal information.

If the conversation contains any such content (even if mixed with benign content), label it 1 (harmful).

## What Counts as Benign (label = 0)
Assign label 0 in all other cases, including when:
- The content is protected speech (opinions, criticism, political views), even if negative or offensive.
- Messages are negative about public figures but are not clearly libelous or fabricated factual claims.
- The assistant summarizes copyrighted text instead of reproducing it verbatim.
- The discussion involves general information about weapons, crime, or sensitive topics without concrete, step-by-step instructions to commit illegal acts.
- The content is sexual or adult in nature but does not involve minors, exploitation, or sex crimes as defined above.
- The conversation is ambiguous, borderline, or could reasonably have a benign interpretation.

When in doubt between harmful and benign, you must choose benign (label = 0).

## Output Behavior
You do not generate new text or rewrite the conversation. You only use this policy to decide whether the correct label should be 0 (benign) or 1 (harmful).
"""

In [ ]:
def load_jsonl(path: Path) -> List[Dict[str, Any]]:
    """Load a JSONL file into a list of dicts."""
    data: List[Dict[str, Any]] = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            data.append(json.loads(line))
    return data


def conversation_to_text(conversation: List[Dict[str, str]]) -> str:
    """Convert a list of {role, message} turns into a plain-text conversation."""
    lines = []
    for turn in conversation:
        role = turn.get("role", "user").capitalize()
        msg = turn.get("message", "").strip()
        lines.append(f"{role}: {msg}")
    return "\n".join(lines)



def build_training_example(example: Dict[str, Any]) -> Dict[str, Any]:
    """Map a ContentID-style example to plain text + binary label."""
    conv = example.get("conversation") or []
    label = int(example.get("label", 0))

    conv_text = conversation_to_text(conv)
    prompt = f"""System: {SYSTEM_CLASSIFIER_PROMPT}

    Input Conversation:
    {conv_text}

    Output: 
    """
    return {"text": prompt, "label": label}


def build_datasets(train_path: Path, val_path: Path | None = None) -> DatasetDict:
    assert train_path.exists(), f"Train JSONL not found: {train_path}"

    train_raw = load_jsonl(train_path)
    train_records = [build_training_example(row) for row in train_raw]

    dataset_dict: Dict[str, Dataset] = {}
    dataset_dict["train"] = Dataset.from_dict(
        {
            "text": [r["text"] for r in train_records],
            "label": [int(r["label"]) for r in train_records],
        }
    )

    if val_path is not None and val_path.exists():
        val_raw = load_jsonl(val_path)
        val_records = [build_training_example(row) for row in val_raw]
        dataset_dict["validation"] = Dataset.from_dict(
            {
                "text": [r["text"] for r in val_records],
                "label": [int(r["label"]) for r in val_records],
            }
        )

    ds = DatasetDict(dataset_dict)
    print(ds)
    return ds


datasets = build_datasets(TRAIN_JSONL, VAL_JSONL if VAL_JSONL.exists() else None)

In [ ]:
# -----------------------------
# Tokenizer, tokenization, and base model (sequence classification)
# -----------------------------

load_kwargs: Dict[str, Any] = {}
if HF_TOKEN is not None and HF_TOKEN != "":
    load_kwargs["token"] = HF_TOKEN

print("Loading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL_NAME,
    use_fast=True,
    **load_kwargs,
)

# Ensure pad token is set
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"


def tokenize_function(batch: Dict[str, List[Any]]) -> Dict[str, Any]:
    encodings = tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
        padding="max_length",
    )
    encodings["labels"] = batch["label"]
    return encodings


print("Tokenizing datasets...")

tokenized_datasets = datasets.map(
    tokenize_function,
    batched=True,
    remove_columns=["text", "label"],
)

quantization_config = None
if USE_4BIT:
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
    )

print("Loading sequence classification base model (this can take a while)...")

config = AutoConfig.from_pretrained(
    BASE_MODEL_NAME,
    num_labels=2,
    problem_type="single_label_classification",
    **load_kwargs,
)

model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL_NAME,
    config=config,
    device_map="auto",
    quantization_config=quantization_config,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    **load_kwargs,
)

print("Model loaded.")

In [ ]:
# -----------------------------
# LoRA config, metrics & Trainer (sequence classification + MLflow)
# -----------------------------

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="SEQ_CLS",
    # These target modules work for most LLaMA/Gemma/SmolLM-style architectures.
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)

model = get_peft_model(model, lora_config)

steps_per_epoch = max(1, len(tokenized_datasets["train"]) // (8 * 4))  # rough heuristic

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=8,
    num_train_epochs=2,
    learning_rate=2e-4,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    logging_steps=10,
    save_steps=steps_per_epoch,
    save_total_limit=3,
    evaluation_strategy="steps" if "validation" in tokenized_datasets else "no",
    eval_steps=steps_per_epoch if "validation" in tokenized_datasets else None,
    bf16=torch.cuda.is_available(),
    gradient_checkpointing=True,
    optim="paged_adamw_8bit" if USE_4BIT else "adamw_torch",
    report_to=["mlflow"],
)


# Metrics: accuracy, F1, ROC-AUC, AUPR, FPR@TPR 90/95
import numpy as np
import evaluate
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve

accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    if isinstance(logits, (tuple, list)):
        logits = logits[0]

    # Convert to numpy
    logits = np.array(logits)
    labels = np.array(labels).astype("int32")

    # Probabilities for harmful class (index 1)
    # Use stable softmax via numpy
    logits_shifted = logits - logits.max(axis=-1, keepdims=True)
    exp_logits = np.exp(logits_shifted)
    probs = exp_logits / exp_logits.sum(axis=-1, keepdims=True)
    harmful_probs = probs[:, 1]

    # Binary predictions at 0.5 threshold on harmful probability
    preds = (harmful_probs >= 0.5).astype("int32")

    metrics = {}
    metrics.update(accuracy_metric.compute(predictions=preds, references=labels))
    metrics.update(f1_metric.compute(predictions=preds, references=labels))

    # ROC-AUC and AUPR
    try:
        roc_auc = roc_auc_score(labels, harmful_probs)
    except ValueError:
        roc_auc = float("nan")

    try:
        aupr = average_precision_score(labels, harmful_probs)
    except ValueError:
        aupr = float("nan")

    # FPR at TPR 0.90 and 0.95
    try:
        fpr, tpr, _ = roc_curve(labels, harmful_probs)
        fpr_at_90 = float(fpr[tpr >= 0.90][0]) if np.any(tpr >= 0.90) else float("nan")
        fpr_at_95 = float(fpr[tpr >= 0.95][0]) if np.any(tpr >= 0.95) else float("nan")
    except ValueError:
        fpr_at_90 = float("nan")
        fpr_at_95 = float("nan")

    metrics.update(
        {
            "roc_auc": roc_auc,
            "aupr": aupr,
            "fpr_at_tpr_90": fpr_at_90,
            "fpr_at_tpr_95": fpr_at_95,
        }
    )

    return metrics


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets.get("validation"),
    tokenizer=tokenizer,
    compute_metrics=compute_metrics if "validation" in tokenized_datasets else None,
)

print("Trainable parameters (after LoRA):")
model.print_trainable_parameters()

In [ ]:
# -----------------------------
# MLflow experiment configuration
# -----------------------------

import mlflow

# Use local file-based tracking under logs/experiments
tracking_uri = MLFLOW_ROOT.as_uri() if "MLFLOW_ROOT" in globals() else None
if tracking_uri is not None:
    mlflow.set_tracking_uri(tracking_uri)

MLFLOW_EXPERIMENT_NAME = "contentid_gemma_smol_lora"
mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)

print(f"MLflow tracking URI: {tracking_uri}")
print(f"MLflow experiment set to: {MLFLOW_EXPERIMENT_NAME}")

In [ ]:
# -----------------------------
# Train & save LoRA adapters (sequence classification)
# -----------------------------

train_result = trainer.train()
print(train_result)

# Optional: final evaluation on validation set (if present)
eval_metrics = {}
if "validation" in tokenized_datasets:
    eval_metrics = trainer.evaluate()
    print("Final eval metrics:", eval_metrics)

# Save adapters + tokenizer to models/...
trainer.model.save_pretrained(MODEL_DIR)
tokenizer.save_pretrained(MODEL_DIR)

# Save train/eval metrics to logs/metrics
metrics_payload = {
    "train": train_result.metrics,
    "eval": eval_metrics,
}
metrics_file = METRICS_ROOT / f"{RUN_NAME}_metrics.json"
with metrics_file.open("w", encoding="utf-8") as f:
    json.dump(metrics_payload, f, indent=2)

print(f"Saved LoRA-tuned classifier to: {MODEL_DIR}")
print(f"Saved metrics to: {metrics_file}")
print("To load later, use AutoModelForSequenceClassification.from_pretrained(..., device_map='auto') on this folder.")

## Inference: get logits, log-probs, sigmoid, and predictions

The next cell defines a helper function that runs the trained `AutoModelForSequenceClassification` and returns raw logits, log-softmax, sigmoid scores, and the final binary prediction for each conversation.

In [ ]:
import torch


def classify_conversations(conversations: List[List[Dict[str, str]]]):
    """Run the trained classifier and return rich outputs for each conversation.

    For each input conversation, returns a dict with:
      - logits: raw scores for labels [0 (benign), 1 (harmful)]
      - log_probs: log-softmax over the two labels
      - softmax_probs: softmax probabilities over labels 0/1
      - harmful_sigmoid: sigmoid applied to the harmful logit (index 1)
      - pred_label: final predicted label (0 or 1)
    """

    prompts: List[str] = []
    for conv in conversations:
        conv_text = conversation_to_text(conv)
        prompt = f"""System: {SYSTEM_CLASSIFIER_PROMPT}

        Conversation:
        {conv_text}

        Question: Is this conversation clearly illegal or egregiously harmful under the above policy?
        Assistant (answer with a single word: 'harmful' or 'benign').
        """
        prompts.append(prompt)

    enc = tokenizer(
        prompts,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
        padding=True,
    ).to(model.device)

    model.eval()
    with torch.no_grad():
        outputs = model(**enc)

    logits = outputs.logits  # [batch, 2]
    log_probs = torch.log_softmax(logits, dim=-1)
    probs = torch.softmax(logits, dim=-1)

    # Treat index 1 as "harmful" logit and apply sigmoid
    harmful_logit = logits[:, 1]
    harmful_sigmoid = torch.sigmoid(harmful_logit)

    preds = torch.argmax(logits, dim=-1)

    results = []
    for i in range(logits.size(0)):
        results.append(
            {
                "logits": logits[i].cpu().tolist(),
                "log_probs": log_probs[i].cpu().tolist(),
                "softmax_probs": probs[i].cpu().tolist(),
                "harmful_sigmoid": float(harmful_sigmoid[i].cpu().item()),
                "pred_label": int(preds[i].cpu().item()),
            }
        )

    return results


# Example usage after training (uncomment and adapt):
# sample = load_jsonl(VAL_JSONL)[:2]
# convs = [ex["conversation"] for ex in sample]
# preds = classify_conversations(convs)
# preds[0]